In [2]:
!pip install -q langchain-text-splitters

In [8]:
import os
import re
import torch
import gc
import pandas as pd
import numpy as np
import json
import pickle
from tqdm import tqdm
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
# from langchain_community.vectorstores import FAISS
# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ====================  TEXT EXTRACTION ====================
def extract_text_from_htm(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            html_content = f.read()

        soup = BeautifulSoup(html_content, 'html.parser')
        raw_text = soup.get_text(separator='\n')

        lines = [line.strip() for line in raw_text.split('\n') if line.strip()]
        text = '\n'.join(lines)
        clean_text = re.sub(r'\n\s*\n+', '\n\n', text).strip()
        return clean_text
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

# jinna5_nano

In [ ]:
books_folder = "/content/drive/MyDrive/books/"
save_folder = "/content/drive/MyDrive/Book_Embeddings/jinaa5_nano"

# model_name = "jinaai/jina-embeddings-v5-text-nano"
model_name = "jinaai/jina-embeddings-v5-text-nano-retrieval"


model = SentenceTransformer(model_name,
                            trust_remote_code=True,
                            device='cuda' if torch.cuda.is_available() else 'cpu'
                            )

# Fixed chunk settings for fair comparison
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=40,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

htm_files = [f for f in os.listdir(books_folder) if f.lower().endswith(('.htm', '.html'))]
actions = []

for file_name in tqdm(htm_files):          # Start with 20 books
    base_name = file_name.replace(".htm", "").replace(".html", "")
    embedding_file = os.path.join(save_folder, f"{base_name}_embeddings.pkl")

    # Skip if already processed
    if os.path.exists(embedding_file):
        print(f"✅ Already processed: {file_name}")
        continue

    file_path = os.path.join(books_folder, file_name)
    text = extract_text_from_htm(file_path)

    # if len(text) < 300:
    #     continue

    # chunks = chunk_text(text)
    chunks = text_splitter.split_text(text)

    embeddings = model.encode(chunks, batch_size=64, show_progress_bar=False)

    # Save to Drive
    data_to_save = {
        "chunks": chunks,
        "embeddings": embeddings,
        "file_name": file_name,
        "book_title": base_name
    }

    with open(embedding_file, "wb") as f:
        pickle.dump(data_to_save, f)

    print(f"💾 Saved embeddings for {file_name}")

    # Clean memory
    del chunks, embeddings
    torch.cuda.empty_cache()
    gc.collect()


Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


model.safetensors:   0%|          | 0.00/424M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/487 [00:00<?, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


...


 88%|████████▊ | 14/16 [09:21<00:15,  7.95s/it]

💾 Saved embeddings for 20790-a-14040108-tarbiyat-altefl.htm



 94%|█████████▍| 15/16 [09:30<00:08,  8.30s/it]

💾 Saved embeddings for 11889-e-13940914-the-roots-of-religion.htm



100%|██████████| 16/16 [09:55<00:00, 37.25s/it]


In [ ]:
books_folder = "/content/drive/MyDrive/books/"
save_folder = "/content/drive/MyDrive/Book_Embeddings_nano"

# model_name = "jinaai/jina-embeddings-v5-text-nano"
model_name = "jinaai/jina-embeddings-v5-text-nano-retrieval"


model = SentenceTransformer(model_name,
                            trust_remote_code=True,
                            device='cuda' if torch.cuda.is_available() else 'cpu'
                            )

# Fixed chunk settings for fair comparison
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

htm_files = [f for f in os.listdir(books_folder) if f.lower().endswith(('.htm', '.html'))]
actions = []

for file_name in tqdm(htm_files):          # Start with 20 books
    base_name = file_name.replace(".htm", "").replace(".html", "")
    embedding_file = os.path.join(save_folder, f"{base_name}_embeddings.pkl")

    # Skip if already processed
    if os.path.exists(embedding_file):
        print(f"✅ Already processed: {file_name}")
        continue

    file_path = os.path.join(books_folder, file_name)
    text = extract_text_from_htm(file_path)

    # if len(text) < 300:
    #     continue

    # chunks = chunk_text(text)
    chunks = text_splitter.split_text(text)

    embeddings = model.encode(chunks, batch_size=64, show_progress_bar=False)

    # Save to Drive
    data_to_save = {
        "chunks": chunks,
        "embeddings": embeddings,
        "file_name": file_name,
        "book_title": base_name
    }

    with open(embedding_file, "wb") as f:
        pickle.dump(data_to_save, f)

    print(f"💾 Saved embeddings for {file_name}")

    # Clean memory
    del chunks, embeddings
    torch.cuda.empty_cache()
    gc.collect()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

 88%|████████▊ | 14/16 [09:46<00:16,  8.21s/it]

💾 Saved embeddings for 20790-a-14040108-tarbiyat-altefl.htm


 94%|█████████▍| 15/16 [09:55<00:08,  8.38s/it]

💾 Saved embeddings for 11889-e-13940914-the-roots-of-religion.htm


100%|██████████| 16/16 [10:22<00:00, 38.91s/it]


# Test model embeddings

In [26]:
for each in os.listdir('/content/drive/MyDrive/books'):
  print(each)

اصول کافی همراه با ترجمه و شرح های مختلف جلد 2 - کتابخانه دیجیتال (بازار کتاب) قائمیه.html
ترجمه وسائل الشيعة الي تحصيل مسائل الشريعة جلد 22 - کتابخانه دیجیتال (بازار کتاب) قائمیه.html
ثواب الأعمال و عقاب الأعمال همراه با 4 ترجمه فارسی - کتابخانه دیجیتال (بازار کتاب) قائمیه.html
سیر مطالعات مهدوی - کتابخانه دیجیتال (بازار کتاب) قائمیه.html
سیر مطالعاتی حج و زیارت - کتابخانه دیجیتال (بازار کتاب) قائمیه.html
سیر مطالعاتی خداشناسی (توحید) - کتابخانه دیجیتال (بازار کتاب) قائمیه.html
سیر مطالعاتی عدل الهی - کتابخانه دیجیتال (بازار کتاب) قائمیه.html
آرزوی سوم ماجرای جنگ خندق -()_9393.html
احد -()_2868.html
تاكتيك های جنگی رسول خدا صلی الله عليه و آله-()_5117.html
حمزه سيد الشهداء عليه السلام -()_3415.html
خيبر شهری كه فتح آن به دست علي (علیه الس... و آله وسلم) را خرسند كرد -()_10816.html
سپاهی كه روانه نشد -()_5093.html
مديريت در بحران ( تحليلي از جنگ خندق يا احزاب ) -()_5108.html
20790-a-14040108-tarbiyat-altefl.htm
11889-e-13940914-the-roots-of-religion.htm


In [29]:
import os
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

# ===================== CONFIG =====================
embeddings_folder = "/content/drive/MyDrive/Book_Embeddings_nano"
# embeddings_folder = "/content/drive/MyDrive/Book_Embeddings/jinaa5_nano"

# ===================== BENCHMARK =====================
with open('/content/drive/MyDrive/output (2).json', 'r', encoding='utf-8-sig') as f:
   original_benchmark = json.load(f)

benchmark = []
for each in original_benchmark:
  if each['expected_book']+'.htm' in os.listdir('/content/drive/MyDrive/books') or each['expected_book']+'.html' in os.listdir('/content/drive/MyDrive/books'):
    benchmark.append(each)

print(f"Loaded {len(benchmark)} benchmark queries\n")

# ===================== LOAD MODEL =====================
print("Loading model for query encoding...")
model = SentenceTransformer(
    "jinaai/jina-embeddings-v5-text-nano-retrieval",
    trust_remote_code=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

if torch.cuda.is_available():
    model = model.half()

# ===================== LOAD ALL EMBEDDINGS =====================
print("Loading embeddings...")
all_chunks = []
all_metadata = []
all_embeddings = []

for file in tqdm(os.listdir(embeddings_folder)):
    if file.endswith("_embeddings.pkl"):
        with open(os.path.join(embeddings_folder, file), "rb") as f:
            data = pickle.load(f)
            for chunk, emb in zip(data["chunks"], data["embeddings"]):
                all_chunks.append(chunk)
                all_metadata.append({
                    "book_title": data["book_title"],
                    "file_name": data["file_name"],
                    "text": chunk
                })
                all_embeddings.append(emb)

all_embeddings = np.array(all_embeddings)
print(f"✅ Loaded {len(all_chunks)} chunks from {len(set(m['book_title'] for m in all_metadata))} books\n")

# ===================== EVALUATE =====================
def evaluate(top_k=5):
    print(f"Evaluating with Top-{top_k} results\n")
    hits = 0
    results = []

    for item in benchmark:
        query = item["query"]
        expected_book = item["expected_book"]

        # Encode query
        query_emb = model.encode([query], normalize_embeddings=True)

        # Get similarities
        similarities = cosine_similarity(query_emb, all_embeddings)[0]
        top_indices = similarities.argsort()[-top_k:][::-1]

        # Check hit
        top_books = [all_metadata[i]["book_title"] for i in top_indices]
        is_hit = expected_book in top_books
        if is_hit:
            hits += 1

        rank = top_books.index(expected_book) + 1 if is_hit else None

        print(f"Query: {query}")
        print(f"Expected: {expected_book[:70]}...")
        print("Top Results:")

        for r, idx in enumerate(top_indices, 1):
            score = similarities[idx]
            book = all_metadata[idx]["book_title"]
            preview = all_metadata[idx]["text"][:160].replace("\n", " ")
            marker = "✅" if book == expected_book else "  "
            print(f"{marker} {r}. [{score:.4f}] {book[:65]}...")

        print("-" * 80)

        results.append({
            "query_id": item["id"],
            "hit": is_hit,
            "rank": rank
        })

    print(f"\n🎯 FINAL RESULT:")
    print(f"Hit@{top_k} = {hits}/{len(benchmark)} ({hits/len(benchmark)*100:.1f}%)")

# Run evaluation
evaluate(top_k=5)

Loaded 20 benchmark queries

Loading model for query encoding...


Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Loading embeddings...


  0%|          | 0/16 [00:00<?, ?it/s]

✅ Loaded 55255 chunks from 16 books

Evaluating with Top-5 results

Query: A child who never tries to find out who raised him
Expected: 11889-e-13940914-the-roots-of-religion...
Top Results:
   1. [0.6095] 20790-a-14040108-tarbiyat-altefl...
   2. [0.6083] 20790-a-14040108-tarbiyat-altefl...
✅ 3. [0.6047] 11889-e-13940914-the-roots-of-religion...
   4. [0.5916] 20790-a-14040108-tarbiyat-altefl...
   5. [0.5800] 20790-a-14040108-tarbiyat-altefl...
--------------------------------------------------------------------------------
Query: A shield that protects us from millions of daily attacks
Expected: 11889-e-13940914-the-roots-of-religion...
Top Results:
✅ 1. [0.4903] 11889-e-13940914-the-roots-of-religion...
   2. [0.4780] تاكتيك های جنگی رسول خدا صلی الله عليه و آله-()_5117l...
   3. [0.4659] تاكتيك های جنگی رسول خدا صلی الله عليه و آله-()_5117l...
   4. [0.4574] تاكتيك های جنگی رسول خدا صلی الله عليه و آله-()_5117l...
   5. [0.4444] اصول کافی همراه با ترجمه و شرح های مختلف جلد 2 - 

In [21]:
from datetime import datetime
# ===================== EVALUATION =====================
results = []
hits_at_5 = 0
output_folder='Evaluation_Results'
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print("Starting Evaluation...\n")

for item in benchmark:
    query = item["query"]
    expected_book = item["expected_book"]

    query_emb = model.encode([query], normalize_embeddings=True)
    similarities = cosine_similarity(query_emb, all_embeddings)[0]
    top_indices = similarities.argsort()[-5:][::-1]   # Top 5

    top_books = [all_metadata[i]["book_title"] for i in top_indices]
    is_hit = expected_book in top_books
    rank = top_books.index(expected_book) + 1 if is_hit else None

    if is_hit:
        hits_at_5 += 1

    # Save detailed result
    results.append({
        "query_id": item["id"],
        "query": query,
        "expected_book": expected_book,
        "expected_concept": item.get("expected_concept", ""),
        "hit@5": is_hit,
        "rank": rank,
        "top1_book": top_books[0],
        "top1_score": float(similarities[top_indices[0]]),
        "top2_book": top_books[1],
        "top3_book": top_books[2]
    })

# ===================== SAVE RESULTS =====================
df = pd.DataFrame(results)

# Save as CSV
csv_path = os.path.join(output_folder, f"evaluation_results_{timestamp}.csv")
df.to_csv(csv_path, index=False, encoding='utf-8-sig')

# Save as JSON
json_path = os.path.join(output_folder, f"evaluation_results_{timestamp}.json")
with open(json_path, "w", encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# ===================== SUMMARY =====================
print("="*70)
print("🎯 EVALUATION SUMMARY")
print("="*70)
print(f"Total Queries     : {len(benchmark)}")
print(f"Hit@5             : {hits_at_5}/{len(benchmark)} ({hits_at_5/len(benchmark)*100:.1f}%)")
print(f"Results saved to  : {output_folder}")
print(f"Files created:")
print(f"   • {os.path.basename(csv_path)}")
print(f"   • {os.path.basename(json_path)}")
print("="*70)

Starting Evaluation...

🎯 EVALUATION SUMMARY
Total Queries     : 170
Hit@5             : 16/170 (9.4%)
Results saved to  : Evaluation_Results
Files created:
   • evaluation_results_20260520_0727.csv
   • evaluation_results_20260520_0727.json


In [23]:
len(all_chunks)

28162

# jinna3

In [ ]:
books_folder = "/content/drive/MyDrive/books/"
save_folder = "/content/drive/MyDrive/Book_Embeddings/jinaa5_nano"

# model_name = "jinaai/jina-embeddings-v5-text-nano"
model_name = "jinaai/jina-embeddings-v5-text-nano-retrieval"


model = SentenceTransformer(model_name,
                            trust_remote_code=True,
                            device='cuda' if torch.cuda.is_available() else 'cpu'
                            )

# Fixed chunk settings for fair comparison
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=40,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

htm_files = [f for f in os.listdir(books_folder) if f.lower().endswith(('.htm', '.html'))]
actions = []

for file_name in tqdm(htm_files):          # Start with 20 books
    base_name = file_name.replace(".htm", "").replace(".html", "")
    embedding_file = os.path.join(save_folder, f"{base_name}_embeddings.pkl")

    # Skip if already processed
    if os.path.exists(embedding_file):
        print(f"✅ Already processed: {file_name}")
        continue

    file_path = os.path.join(books_folder, file_name)
    text = extract_text_from_htm(file_path)

    # if len(text) < 300:
    #     continue

    # chunks = chunk_text(text)
    chunks = text_splitter.split_text(text)

    embeddings = model.encode(chunks, batch_size=64, show_progress_bar=False)

    # Save to Drive
    data_to_save = {
        "chunks": chunks,
        "embeddings": embeddings,
        "file_name": file_name,
        "book_title": base_name
    }

    with open(embedding_file, "wb") as f:
        pickle.dump(data_to_save, f)

    print(f"💾 Saved embeddings for {file_name}")

    # Clean memory
    del chunks, embeddings
    torch.cuda.empty_cache()
    gc.collect()


# jina3 embeddings

## different chunk sizes



*   chunk_size = 250
*   chunk_size =

